In [6]:
import re
import pandas as pd
from pathlib import Path

# --- Конфигурация ---
INPUT_DIR = Path("/Users/haraka/PycharmProjects/study/social/001_1_3_Определение_объёма_эмпирического_материала/data_assembler/sudact_cases")
OUTPUT_DIR = Path("/Users/haraka/PycharmProjects/study/social/001_1_3_Определение_объёма_эмпирического_материала/data_stucture_generator/result")
# -------------------

def parse_sudact_file(filepath):
    """
    Парсит один файл и возвращает словарь с данными.
    Если данные не найдены, возвращает None или словарь с пропусками.
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
    except Exception as e:
        print(f"Ошибка чтения файла {filepath}: {e}")
        return None

    # Убираем лишние символы (многоточия, звёздочки), которые мешают регуляркам
    # Но делаем это аккуратно, чтобы не сломать номера
    text_clean = re.sub(r'\*{3,}', ' ', text) # Заменяем *** на пробел
    text_clean = re.sub(r'_{2,}', ' ', text_clean) # Заменяем ___ на пробел

    data = {
        'file_name': filepath.name,
        'doc_type': None,
        'date_decision': None,
        'case_number': None,
        'court_name': None,
        'judge_name': None,
        'defendant_name': None,
        'defendant_inn': None,
        'defendant_ogrn': None,
        'defendant_address': None,
        'article': None,
        'violation_essence': None,
        'inspector_name': None,
        'order_number': None,
        'order_date': None,
        'deadline_date': None,
        'verdict': None,
        'penalty_type': None,
        'penalty_amount': None,
        'payment_details': None,
    }

    # 1. Основные заголовки
    match = re.search(r'(Постановление|Решение)\s+от\s+(\d+\s+\S+\s+\d{4}\s*г\.?)', text, re.IGNORECASE)
    if match:
        data['doc_type'] = match.group(1).strip()
        data['date_decision'] = match.group(2).strip()
    else:
        # fallback
        data['doc_type'] = "Постановление"

    # 2. Номер дела (встречается в разных местах)
    match = re.search(r'делу\s+№?\s*([\w\-/]+)', text, re.IGNORECASE)
    if match:
        data['case_number'] = match.group(1).strip()
    else:
        # Ищем в формате "ДЕЛО № 5-***/2017-145"
        match = re.search(r'ДЕЛО\s+№\s*([\w\-/]+)', text)
        if match:
            data['case_number'] = match.group(1).strip()

    # 3. Мировой судья
    match = re.search(r'Мировой судья\s+([^,\n]+)', text)
    if match:
        data['judge_name'] = match.group(1).strip()
    else:
        # Альтернативный паттерн
        match = re.search(r'Мировой судья\s+судебного участка\s+№\d+\s+([А-Я][а-я]+\s+[А-Я]\.?[А-Я]\.?)', text)
        if match:
            data['judge_name'] = match.group(1).strip()

    # 4. Суд
    match = re.search(r'Судебный участок №\s*\d+\s*\(([^)]+)\)', text)
    if match:
        data['court_name'] = f"Судебный участок ({match.group(1).strip()})"
    else:
        match = re.search(r'(Мировой судья судебного участка №\s*\d+\s+[^,\n]+)', text)
        if match:
            data['court_name'] = match.group(1).strip()

    # 5. Ответчик (Юрлицо)
    match = re.search(r'в отношении\s+([^,]+(?:ТСЖ|ООО|ОАО|ЗАО)[^,]+)', text, re.IGNORECASE)
    if match:
        data['defendant_name'] = match.group(1).strip()
    else:
        match = re.search(r'(ТСЖ|ООО|ОАО|ЗАО)\s+[«"]([^»"]+)[»"]', text, re.IGNORECASE)
        if match:
            data['defendant_name'] = match.group(0).strip()

    # 6. ИНН и ОГРН
    match = re.search(r'ИНН\s+([\d]+)', text)
    if match:
        data['defendant_inn'] = match.group(1).strip()
    match = re.search(r'ОГРН\s+([\d]+)', text)
    if match:
        data['defendant_ogrn'] = match.group(1).strip()

    # 7. Адрес
    match = re.search(r'адрес:\s*([^,]+(?:Санкт-Петербург|Москва|обл\.)[^,]+)', text, re.IGNORECASE)
    if match:
        data['defendant_address'] = match.group(1).strip()

    # 8. Инспекция и Предписание
    match = re.search(r'(Государственной жилищной инспекции|Роспотребнадзор|Прокуратура|МЧС)', text, re.IGNORECASE)
    if match:
        data['inspector_name'] = match.group(1).strip()

    # 9. Номер предписания
    match = re.search(r'предписание\s+№?\s*([\w\-]+)\s+от\s+(\d+\s+\S+\s+\d{4}\s*г\.?)', text, re.IGNORECASE)
    if match:
        data['order_number'] = match.group(1).strip()
        data['order_date'] = match.group(2).strip()
    else:
        # fallback для номера
        match = re.search(r'предписание\s+№?\s*([\w\-]+)', text, re.IGNORECASE)
        if match:
            data['order_number'] = match.group(1).strip()

    # 10. Срок исполнения (дедлайн)
    match = re.search(r'срок\s+(?:выполнения|исполнения|устранения)(?:\s+нарушений)?\s+[–—]?\s*до\s+([^.\n]+?г\.?)', text, re.IGNORECASE)
    if match:
        data['deadline_date'] = match.group(1).strip()
    else:
        match = re.search(r'срок\s+[–—]\s+([^.\n]+?включительно)', text, re.IGNORECASE)
        if match:
            data['deadline_date'] = match.group(1).strip()

    # 11. Суть нарушения
    # Берем предложение после "установлено, что"
    match = re.search(r'установлено, что\s+([^\.]+(?:не выполнило|не исполнило)[^\.]+)\.', text, re.IGNORECASE)
    if match:
        data['violation_essence'] = match.group(1).strip() + "."

    # 12. Статья
    match = re.search(r'ст\.?\s*([\d\.]+)\s+КоАП', text, re.IGNORECASE)
    if match:
        data['article'] = "ст. " + match.group(1).strip() + " КоАП РФ"

    # 13. Вердикт и Наказание
    if "признать виновным" in text.lower():
        data['verdict'] = "Виновен"
    elif "прекратить" in text.lower():
        data['verdict'] = "Прекращено"
    else:
        data['verdict'] = "Не определено"

    # 14. Штраф
    match = re.search(r'штрафа\s+в\s+размере\s+([\d\s]+)\s+руб', text, re.IGNORECASE)
    if match:
        data['penalty_amount'] = match.group(1).strip().replace(' ', '')
        data['penalty_type'] = "Штраф"
    else:
        # Ищем в резолютивной части
        match = re.search(r'назначить\s+[^\.]+?\s+(\d+)\s+руб', text, re.IGNORECASE)
        if match:
            data['penalty_amount'] = match.group(1).strip()
            data['penalty_type'] = "Штраф"

    # 15. Платежные реквизиты (собираем блок)
    payment_block = re.search(r'(?:реквизитам|перечисляется)[:\s]+(.*?)(?:\n\s*\n|Постановление может быть обжаловано)', text, re.DOTALL | re.IGNORECASE)
    if payment_block:
        # Очищаем от лишних пробелов
        data['payment_details'] = ' '.join(payment_block.group(1).split()).strip()
    else:
        # Если не нашли блок, ищем УИН
        match = re.search(r'(УИН\s+[\d]+)', text)
        if match:
            data['payment_details'] = match.group(1)

    return data

def main():
    """Главная функция: обходит папку и создает DataFrame."""
    if not INPUT_DIR.exists():
        print(f"Ошибка: Директория {INPUT_DIR} не найдена.")
        return

    all_cases_data = []
    txt_files = list(INPUT_DIR.glob("*.txt"))

    if not txt_files:
        print(f"В директории {INPUT_DIR} не найдено .txt файлов.")
        return

    print(f"Найдено файлов: {len(txt_files)}. Начинаю парсинг...")

    for i, file_path in enumerate(txt_files):
        # Прогресс
        if (i + 1) % 10 == 0:
            print(f"Обработано {i + 1}/{len(txt_files)} файлов...")

        parsed_data = parse_sudact_file(file_path)
        if parsed_data:
            all_cases_data.append(parsed_data)

    # Создаем DataFrame
    df = pd.DataFrame(all_cases_data)

    print("\nПарсинг завершен.")
    print(f"Успешно обработано файлов: {len(df)} из {len(txt_files)}")

    # --- Анализ и сохранение ---
    if not df.empty:
        print("\nПервые 5 строк датафрейма:")
        print(df.head())

        print("\nИнформация о колонках и пропусках:")
        print(df.info())

        # Сохраняем в CSV для дальнейшей работы в Excel или любом другом редакторе
        output_csv = OUTPUT_DIR / "parsed_cases_data.csv"
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"\nДанные сохранены в файл: {output_csv}")

        # Дополнительно сохраняем в Excel, если нужно
        try:
            output_excel = OUTPUT_DIR / "parsed_cases_data.xlsx"
            df.to_excel(output_excel, index=False)
            print(f"Данные сохранены в файл: {output_excel}")
        except ImportError:
            print("Для сохранения в Excel установите библиотеку openpyxl: pip install openpyxl")
    else:
        print("DataFrame пуст. Проверьте регулярные выражения на первом файле.")

if __name__ == "__main__":
    # Убедитесь, что у вас установлены pandas и openpyxl
    # pip install pandas openpyxl
    main()

Найдено файлов: 276. Начинаю парсинг...
Обработано 10/276 файлов...
Обработано 20/276 файлов...
Обработано 30/276 файлов...
Обработано 40/276 файлов...
Обработано 50/276 файлов...
Обработано 60/276 файлов...
Обработано 70/276 файлов...
Обработано 80/276 файлов...
Обработано 90/276 файлов...
Обработано 100/276 файлов...
Обработано 110/276 файлов...
Обработано 120/276 файлов...
Обработано 130/276 файлов...
Обработано 140/276 файлов...
Обработано 150/276 файлов...
Обработано 160/276 файлов...
Обработано 170/276 файлов...
Обработано 180/276 файлов...
Обработано 190/276 файлов...
Обработано 200/276 файлов...
Обработано 210/276 файлов...
Обработано 220/276 файлов...
Обработано 230/276 файлов...
Обработано 240/276 файлов...
Обработано 250/276 файлов...
Обработано 260/276 файлов...
Обработано 270/276 файлов...

Парсинг завершен.
Успешно обработано файлов: 276 из 276

Первые 5 строк датафрейма:
                file_name       doc_type       date_decision     case_number  \
0   7cRbAvt7vtYq_5-51